# Module 06 — Web Search

Give **`claude-opus-4-8`** the server-side **`web_search`** tool so it answers with **real-time information and citations** — all through the Agent Platform.

Web search is **Anthropic-executed**: the platform runs the search for you, so there is **no local tool loop** (unlike Module 04 function calling). This builds on **Modules 00–05** and uses the same **`AnthropicVertex` / ADC** path.

## Part B — Bootstrap

Setup mirrors Module 00 — see that module for full detail (prerequisites, ADC, logging).

In [ ]:
%pip install --quiet "anthropic[vertex]"

In [ ]:
# ===== CONDENSED BOOTSTRAP (mirrors Module 00) =====
import subprocess
from anthropic import AnthropicVertex

def _default_project() -> str:
    try:
        out = subprocess.run(
            ["gcloud", "config", "get-value", "project"],
            capture_output=True, text=True, timeout=15,
        ).stdout.strip()
        if out and out != "(unset)":
            return out
    except Exception:
        pass
    return "<YOUR_PROJECT_ID>"

PROJECT_ID = _default_project()   # or hard-code: PROJECT_ID = "my-project-id"
LOCATION   = "global"
MODEL      = "claude-opus-4-8"

assert PROJECT_ID and PROJECT_ID != "<YOUR_PROJECT_ID>", (
    "PROJECT_ID is still the placeholder. Run `gcloud config set project <id>` "
    "so it auto-detects, or edit PROJECT_ID directly."
)

client = AnthropicVertex(project_id=PROJECT_ID, region=LOCATION)
print(f"✅ Client ready (project={PROJECT_ID}, region={LOCATION}, model={MODEL}).")

## Part C — Enable web search

Web search is a **server tool** — Anthropic runs the search and feeds results back to the model in the same call. You just declare the tool; you never execute anything locally.

**Tool definition:**
```python
{"type": "web_search_20250305", "name": "web_search", "max_uses": 5}
```
**Optional params:** `max_uses`, `allowed_domains`, `blocked_domains`, and `user_location` (`{"type": "approximate", "city", "region", "country", "timezone"}`).

> If a call errors saying the tool isn't enabled, web search may need to be **enabled for your org/project** first — confirm in the docs linked at the end.

In [ ]:
resp = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 5}],
    messages=[{
        "role": "user",
        "content": "What are the latest Claude models available on Google Cloud's Agent Platform? Cite your sources.",
    }],
)

# Print only the final text (join the text blocks).
print("".join(b.text for b in resp.content if b.type == "text"))

## Part D — Inspect the search activity

The response `content` can contain three relevant block types:

- **`server_tool_use`** — the search query Claude ran (in `block.input`).
- **`web_search_tool_result`** — the results; `block.content` is a list of items with `.title` and `.url`.
- **`text`** — the final answer, with citations.

We guard all attribute access so unexpected block shapes don't crash the cell.

In [ ]:
print("===== SEARCH QUERIES =====")
for block in resp.content:
    if getattr(block, "type", None) == "server_tool_use":
        query = getattr(block, "input", {})
        print(" ", query.get("query", query) if hasattr(query, "get") else query)

print("\n===== RESULT SOURCES =====")
for block in resp.content:
    if getattr(block, "type", None) == "web_search_tool_result":
        items = getattr(block, "content", []) or []
        if not isinstance(items, list):
            continue
        for item in items:
            title = getattr(item, "title", None)
            url = getattr(item, "url", None)
            if title or url:
                print(f"  - {title} | {url}")

print("\n===== FINAL ANSWER =====")
print("".join(b.text for b in resp.content if getattr(b, "type", None) == "text"))

## Part E — Controls

You can shape search behavior:

- **`max_uses`** — caps the number of searches per call (controls cost).
- **`allowed_domains` / `blocked_domains`** — restrict which sources are used.
- **`user_location`** — localizes results to a place/timezone.

In [ ]:
resp2 = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=[{
        "type": "web_search_20250305",
        "name": "web_search",
        "max_uses": 2,
        "allowed_domains": ["docs.claude.com"],
    }],
    messages=[{
        "role": "user",
        "content": "According to the Claude docs, how do I enable extended thinking? Cite the page.",
    }],
)

print("".join(b.text for b in resp2.content if getattr(b, "type", None) == "text"))

## Part F — Notes & recap

### Notes

- **Server tool:** no local execution — the platform runs the search inside the call (contrast Module 04's local tool loop).
- **Citations:** answers include source citations so claims are traceable.
- **Cost:** each search has a cost — use **`max_uses`** to cap it.
- **Versions:** `web_search_20250305` is the basic tool used here; **`web_search_20260209`** adds **dynamic filtering with Opus 4.8** — consider it once you've confirmed availability.
- Docs: https://docs.claude.com/en/docs/agents-and-tools/tool-use/web-search-tool and https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/partner-models/claude/web-search — confirm availability and current limits there, as they may change.

### Recap

- `web_search` is an **Anthropic-executed server tool** — declare it and the platform runs the search; **no local loop**.
- Define it with `{"type": "web_search_20250305", "name": "web_search", "max_uses": 5}`.
- Inspect **`server_tool_use`** (queries), **`web_search_tool_result`** (sources), and **`text`** (cited answer) blocks.
- Control results with `max_uses`, `allowed_domains` / `blocked_domains`, and `user_location`.

**Next: Module 07 — Computer use.**